# Iris Flower Analysis\nResearch question: Is mean petal length different among the three Iris species? Run all cells from top to bottom.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from scipy import stats
from scipy.stats import levene
from statsmodels.stats.multicomp import pairwise_tukeyhsd


## 1. Load and validate data

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.rename(columns={
    "petal length (cm)": "petal_length",
    "target": "species_code"
}).copy()
df["species"] = df["species_code"].map(dict(enumerate(iris.target_names)))
required = {"species", "petal_length"}
if not required.issubset(df.columns):
    raise ValueError(f"Missing columns: {required - set(df.columns)}")
before = len(df)
df["petal_length"] = pd.to_numeric(df["petal_length"], errors="coerce")
df = df[df["species"].isin(["setosa", "versicolor", "virginica"])]
df = df.dropna(subset=["species", "petal_length"]).copy()
print("Rows before:", before, "| Rows included:", len(df))
print("Missing required values:\n", df[["species", "petal_length"]].isna().sum())
print("Exact duplicate rows:", int(df.duplicated().sum()))
print(df["species"].value_counts())


## 2. Descriptive statistics

In [ ]:
summary = df.groupby("species")["petal_length"].agg(
    n="count", mean="mean", std="std", median="median",
    q1=lambda x: x.quantile(.25), q3=lambda x: x.quantile(.75)
)
display(summary)


## 3. Plot

In [ ]:
order = ["setosa", "versicolor", "virginica"]
groups = [df.loc[df.species == s, "petal_length"] for s in order]
plt.figure(figsize=(8,5))
plt.boxplot(groups, labels=order)
plt.xlabel("Species"); plt.ylabel("Petal length (cm)")
plt.title("Petal length by Iris species"); plt.tight_layout(); plt.show()


## 4. Primary test: one-way ANOVA

In [ ]:
anova = stats.f_oneway(*groups)
k, n = len(groups), len(df)
grand = df.petal_length.mean()
ss_between = sum(len(g) * (g.mean()-grand)**2 for g in groups)
ss_total = ((df.petal_length-grand)**2).sum()
eta_sq = ss_between / ss_total
print(f"F({k-1}, {n-k}) = {anova.statistic:.4f}")
print(f"p-value = {anova.pvalue:.6g}")
print(f"eta-squared = {eta_sq:.4f}")


## 5. Diagnostics and post-hoc analysis

In [ ]:
lev = levene(*groups, center="median")
print("Levene statistic:", lev.statistic, "p-value:", lev.pvalue)
fitted = df.groupby("species").petal_length.transform("mean")
residuals = df.petal_length - fitted
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("ANOVA residual Q-Q plot"); plt.tight_layout(); plt.show()
if anova.pvalue < 0.05:
    print(pairwise_tukeyhsd(df.petal_length, df.species, alpha=0.05))
else:
    print("Overall ANOVA not significant; no confirmatory Tukey test.")


## 6. Robustness check

In [ ]:
kw = stats.kruskal(*groups)
print("Kruskal-Wallis H:", kw.statistic, "p-value:", kw.pvalue)


## 7. Interpretation

Use the actual outputs above to complete `reports/research_report.md`. A significant result does not establish causation. Report effect size and diagnostics as well as p-values.